# Hydrology Lab — Environmental Science for Sustainable Development

Notebook outline based on `water lab-1.pdf`. Complete the analysis in Python using the supplied `.mat` data and the logic in `watershedsearch.m` and `stepdown.m`. Questions below are taken from the lab handout; responses and Python code are left open.

**Plotting reminder:** Title plots, label axes, and include colorbars and legends where applicable. MATLAB uses one-based indexing; check index and axis order when translating to Python.

## Setup and data files

Use `australia_rain.mat`, `bangladesh_topography.mat`, `watershedsearch.m`, and `stepdown.m` from this lab folder. Load the data and inspect variable names, shapes, coordinate order, and missing values.

Run Jupyter from this lab folder, or set `LAB3_DATA_DIR` to the folder containing the two `.mat` files. The Python examples use NumPy zero-based indexing.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle
from scipy.io import loadmat
from scipy import stats

# Run Jupyter from the lab folder, or set LAB3_DATA_DIR to that folder.
data_dir = Path(os.environ.get("LAB3_DATA_DIR", ".")).expanduser().resolve()
required = ["australia_rain.mat", "bangladesh_topography.mat"]
missing = [name for name in required if not (data_dir / name).is_file()]
if missing:
    raise FileNotFoundError(f"Missing {missing} in {data_dir}; set LAB3_DATA_DIR")

australia = loadmat(data_dir / "australia_rain.mat")
rainfall = australia["rainfall"]  # lat × lon × year × month
lat = australia["lat"].ravel()
lon = australia["lon"].ravel()
year = australia["year"].ravel()
print(australia["key"].item())
print("rainfall:", rainfall.shape, "lat/lon/year:", lat.shape, lon.shape, year.shape)


# 1. Australian rainfall statistics

## 1.1 Higher dimension arrays

Create same-sized arrays `A` and `B`, stack them in a third dimension as `C`, and explore indexing, a fourth dimension, and dimension removal (MATLAB `squeeze`).

In [ ]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])
C = np.stack((A, B), axis=2)  # MATLAB cat(3,A,B)
K = np.arange(1000).reshape(10, 10, 10)
print("C shape:", C.shape, "layers:\n", C[:, :, 0], "\n", C[:, :, 1])
print("K elements:", K.size, "vertices:", K[0, 0, 0], K[-1, -1, -1])
print("Eight central entries:\n", K[4:6, 4:6, 4:6])
D = np.stack((C, 3 * C), axis=3)  # MATLAB cat(4,C,3*C)
print("D shape:", D.shape)
try:
    np.stack((C, C[:, :-1, :]), axis=3)
except ValueError as exc:
    print("Mismatched dimensions:", exc)
g = C[0, 0, :]  # NumPy drops the two indexed axes automatically
print("g shape:", g.shape, "g:", g)


### Questions: array dimensions and indexing

> How does Matlab display a 3D array? If you had a 10 by 10 by 10 array K, how many elements would it have? What command would you use to see only the entry at a vertex? give two examples. What about the command to look at the eight elements in the very center of the array? Draw a diagram of where these eight entries are in the array.

**Response:**

### Question: incompatible array shapes

> Try cat(4, C, 3*C). Try it with another array that is not the same size. What happens? Explain why.

**Response:**

## 1.2 Data: spatial variation

Load `australia_rain.mat`; inspect `key`, `rainfall`, `lat`, `lon`, and `year`. Check whether longitude belongs on the x-axis and latitude on the y-axis. Plot January 1970 with geographic axes and a colorbar.

In [ ]:
def rain_map(ax, field, title, *, levels=None, vmin=None, vmax=None, cmap="viridis",
             colorbar_label="Rainfall (dataset units)"):
    # Array order is (latitude, longitude); plotting takes (x=lon, y=lat).
    mesh = ax.pcolormesh(lon, lat, field, shading="auto", cmap=cmap,
                         vmin=vmin, vmax=vmax)
    ax.set(title=title, xlabel="Longitude (°E)", ylabel="Latitude (°)")
    plt.colorbar(mesh, ax=ax, label=colorbar_label)
    if levels is not None:
        contours = ax.contour(lon, lat, field, levels=levels, colors="black", linewidths=0.6)
        ax.clabel(contours, inline=True, fontsize=8)  # Extra credit 1
    return mesh

jan_1970 = rainfall[:, :, np.flatnonzero(year == 1970)[0], 0]
fig, ax = plt.subplots(figsize=(9, 6))
rain_map(ax, jan_1970, "January 1970 rainfall", levels=6)
plt.show()


### Extra credit, part 1

> Figure out how to label your contours with numbers placed on the contours (like most contour maps).

**Response / Python approach:**

### January and June climatologies

Compute the mean January field across years and the corresponding June field. Plot both with titles, axes, and colorbars.

In [ ]:
jan_climatology = np.nanmean(rainfall[:, :, :, 0], axis=2)
jun_climatology = np.nanmean(rainfall[:, :, :, 5], axis=2)
fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
rain_map(axes[0], jan_climatology, "January climatology")
rain_map(axes[1], jun_climatology, "June climatology")
plt.show()


### Questions: monthly rainfall

> How does the rainfall in 1970 compare with the average January climatology (the mean January field across all years)? Compare the climatologies for January and June.

**Response:**

### Annual climatology, 2006 rainfall, and anomalies

Average over both year and month for annual climatology. Find 2006 using `year`; plot 2006 and the climatology on a common color scale. Then plot absolute and relative 2006 anomalies. Include 2006 in the climatology.

In [ ]:
# The year axis includes 2006 in both climatologies.
annual_by_year = np.nanmean(rainfall, axis=3)
annual_climatology = np.nanmean(annual_by_year, axis=2)
index_2006 = np.flatnonzero(year == 2006)[0]
assert index_2006.size == 1
rain_2006 = annual_by_year[:, :, index_2006.item()]
common_max = np.nanmax((np.nanmax(annual_climatology), np.nanmax(rain_2006)))
fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
rain_map(axes[0], annual_climatology, "Annual climatology", vmin=0, vmax=common_max)
rain_map(axes[1], rain_2006, "2006 annual mean rainfall", vmin=0, vmax=common_max)
plt.show()

absolute_anomaly = rain_2006 - annual_climatology
relative_anomaly = np.divide(absolute_anomaly, annual_climatology,
                             out=np.full_like(absolute_anomaly, np.nan),
                             where=annual_climatology > 0)
fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
abs_limit = np.nanmax(np.abs(absolute_anomaly))
rel_limit = np.nanmax(np.abs(relative_anomaly))
rain_map(axes[0], absolute_anomaly, "2006 absolute anomaly",
         vmin=-abs_limit, vmax=abs_limit, cmap="RdBu",
         colorbar_label="Rainfall anomaly (dataset units)")
rain_map(axes[1], relative_anomaly, "2006 relative anomaly",
         vmin=-rel_limit, vmax=rel_limit, cmap="RdBu",
         colorbar_label="Fraction of climatology")
plt.show()

index_1987 = np.flatnonzero(year == 1987)[0]
assert index_1987.size == 1
fig, ax = plt.subplots(figsize=(9, 6))
rain_map(ax, annual_by_year[:, :, index_1987.item()], "1987 annual mean rainfall")
plt.show()


### Questions: 2006 drought

> How does the 2006 rainfall compare to the average climatology of the period?

**Response:**

> Which regions in Australia were hardest hit by the drought (in absolute rainfall terms)? What about as a fraction of the climatology?

**Response:**

## 1.3 Statistics: inter-temporal variation

Plot annual climatology with a continent boundary mask; mark and label Darwin and Melbourne. Build annual rainfall series for both cities and check their lengths against `year`.

In [ ]:
# Nearest grid cells to city centers, in (latitude, longitude) order.
city_coords = {"Darwin": (-12.46, 130.84), "Melbourne": (-37.81, 144.96)}
city_indices = {name: (np.abs(lat - latitude).argmin(), np.abs(lon - longitude).argmin())
                for name, (latitude, longitude) in city_coords.items()}
darwin = annual_by_year[city_indices["Darwin"]].reshape(-1)
melbourne = annual_by_year[city_indices["Melbourne"]].reshape(-1)
assert darwin.shape == melbourne.shape == year.shape
print("Nearest grid cells:", {name: (lat[i], lon[j]) for name, (i, j) in city_indices.items()})

fig, ax = plt.subplots(figsize=(9, 6))
rain_map(ax, annual_climatology, "Annual climatology and city locations")
land_mask = np.isfinite(annual_climatology) & (annual_climatology > 0)
ax.contour(lon, lat, land_mask.astype(float), levels=[0.5], colors="black")
for name, (latitude, longitude) in city_coords.items():
    ax.plot(longitude, latitude, "k+", markersize=10)
    ax.text(longitude + 0.5, latitude, name, color="black")
plt.show()


### Question: city climates

> Which city is in a drier climate?

**Response:**

### Correlation and ordinary least squares

Plot both annual series, annotate their correlation, and estimate Darwin rainfall from Melbourne rainfall with an intercept. Compare a library OLS fit with a manual calculation.

In [ ]:
valid = np.isfinite(darwin) & np.isfinite(melbourne)
x = melbourne[valid]
y = darwin[valid]
year_valid = year[valid]
correlation = np.corrcoef(x, y)[0, 1]
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(year_valid, y, label="Darwin")
ax.plot(year_valid, x, label="Melbourne")
ax.set(title=f"Annual mean rainfall (r = {correlation:.3f})", xlabel="Year",
       ylabel="Rainfall (dataset units)")
ax.legend()
plt.show()

X = np.column_stack((np.ones_like(x), x))
betas_library = stats.linregress(x, y)
betas_manual = np.linalg.solve(X.T @ X, X.T @ y)
print("Library [intercept, slope]:", betas_library.intercept, betas_library.slope)
print("Manual  [intercept, slope]:", betas_manual)
np.testing.assert_allclose(betas_manual, [betas_library.intercept, betas_library.slope])
print("Slope standard error (extra credit):", betas_library.stderr)


### Questions: regression

> Check that beta matlab and beta manual are the same and report the coefficients. Comment on the estimate and your interpretation of it.

**Response:**

### Extra credit, part 2

> See help regress and figure out how to quantify the uncertainty in the estimate. Report the commands you use.

**Response / Python commands:**

### Rainfall distributions

Plot Melbourne's annual rainfall histogram. Fit Poisson and normal distributions, display each fitted distribution against the histogram on a compatible scale, and compare the fits.

In [ ]:
x = melbourne[np.isfinite(melbourne)]
lambda_hat = x.mean()  # Poisson MLE; the model assumes integer counts.
mu_hat, sigma_hat = stats.norm.fit(x)
support = np.arange(max(0, int(np.floor(x.min()))), int(np.ceil(x.max())) + 1)
smooth_support = np.linspace(x.min(), x.max(), 500)
fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
for ax in axes:
    ax.hist(x, bins=12, density=True, color="lightgray", edgecolor="black", label="Observed")
    ax.set(xlabel="Annual mean rainfall (dataset units)", ylabel="Density")
axes[0].plot(support, stats.poisson.pmf(support, lambda_hat), label="Poisson PMF")
axes[0].set_title(f"Poisson fit (λ = {lambda_hat:.1f})")
axes[1].plot(smooth_support, stats.norm.pdf(smooth_support, mu_hat, sigma_hat),
             label="Normal PDF")
axes[1].set_title(f"Normal fit (μ = {mu_hat:.1f}, σ = {sigma_hat:.1f})")
for ax in axes:
    ax.legend()
plt.show()


### Questions: distribution fit

> What kind of distribution does this look like to you?

**Response:**

> Does Melbourne’s annual rainfall look like it’s distributed like a Poisson random variable?

**Response:**

> Compare your two plots and comment on which parametric distribution (normal or Poisson) better characterizes the inter-temporal variation in annual rainfall over Melbourne.

**Response:**

# 2. A watershed in Bangladesh

## 2.1 Data

Load `bangladesh_topography.mat` and inspect `topo`. Reproduce the two nested crops in Python, accounting for MATLAB's inclusive, one-based slices. Plot all three maps side by side, orient north upward, and draw rectangles around the cropped regions.

In [ ]:
topo = loadmat(data_dir / "bangladesh_topography.mat")["topo"]
# MATLAB topo(50:100,120:180), then topo2(25:43,10:30).
topo2 = topo[49:100, 119:180]
topo3 = topo2[24:43, 9:30]
assert topo2.shape == (51, 61) and topo3.shape == (19, 21)
fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
for ax, field, title in zip(axes, (topo, topo2, topo3), ("topo", "topo2", "topo3")):
    image = ax.imshow(field, origin="lower", cmap="terrain", interpolation="nearest")
    ax.set(title=title, xlabel="Column (zero-based)", ylabel="Row (zero-based)")
    ax.set_aspect("equal")
    plt.colorbar(image, ax=ax, label="Elevation (m)")
axes[0].add_patch(Rectangle((118.5, 48.5), 61, 51, fill=False, edgecolor="red", linewidth=2))
axes[1].add_patch(Rectangle((8.5, 23.5), 21, 19, fill=False, edgecolor="red", linewidth=2))
plt.show()


## 2.2 Modeling downhill flow

Translate the supplied `stepdown.m` function and `watershedsearch.m` script to Python. Run the watershed search on `topo3`, inspect the `shed` mask, and overlay it on topography. Estimate watershed area from the approximately 1 km × 1 km grid cells and convert to square miles.

The original `stepdown.m` chooses the first minimum in column-first order. Flat integer elevations can create cycles, so the Python translation detects and excludes cyclic paths; inspect the reported count before interpreting the mask.

In [ ]:
def stepdown(neighborhood):
    """Return (row, column) offsets matching stepdown.m's column-first tie order."""
    if neighborhood.shape != (3, 3):
        raise ValueError("stepdown requires a 3 x 3 neighborhood")
    min_rows = np.argmin(neighborhood, axis=0)
    min_values = neighborhood[min_rows, np.arange(3)]
    col = int(np.argmin(min_values))
    return int(min_rows[col] - 1), col - 1


def watershed_search(elevation):
    """Follow each cell to the specified MATLAB floodpoint; break tie cycles."""
    padded = np.pad(elevation, 1, mode="constant", constant_values=101)
    target = (8, 19)  # MATLAB topo4(9,20), converted to zero-based indexes.
    shed = np.zeros(elevation.shape, dtype=np.uint8)
    cycles = 0
    for row in range(1, padded.shape[0] - 1):
        for col in range(1, padded.shape[1] - 1):
            current = (row, col)
            visited = set()
            while current not in visited:
                visited.add(current)
                r, c = current
                dr, dc = stepdown(padded[r - 1:r + 2, c - 1:c + 2])
                if (dr, dc) == (0, 0):
                    break
                current = (r + dr, c + dc)
            else:
                cycles += 1
                continue
            if current == target:
                shed[row - 1, col - 1] = 1
    return shed, cycles


assert stepdown(np.array([[1, 1, 1], [1, 0, 1], [-1, 2, 3]])) == (1, -1)
shed, cycle_count = watershed_search(topo3)
area_km2 = int(shed.sum())  # Each DEM cell is approximately 1 km².
area_mi2 = area_km2 * 0.3861021585
print(f"Watershed: {area_km2} cells ≈ {area_km2} km² ≈ {area_mi2:.2f} mi²")
print("Paths stopped on elevation-tie cycles:", cycle_count)
fig, ax = plt.subplots(figsize=(9, 5))
image = ax.imshow(topo3, origin="lower", cmap="terrain", interpolation="nearest")
ax.contour(shed, levels=[0.5], colors="blue", origin="lower")
ax.plot(18, 7, "r+", markersize=12, label="Floodpoint")
ax.set(title="Watershed boundary over topography", xlabel="Column", ylabel="Row")
ax.legend()
plt.colorbar(image, ax=ax, label="Elevation (m)")
plt.show()


### Questions: watershed output

> Run watershedsearch, it should produce a new matrix shed. Examine this matrix, does it look right?

**Response:**

> Now figure out how many square miles the watershed is.

**Response:**

## 2.3 A monsoon

### Uniform rainfall and floodpoint capacity

Assume uniform rain over `topo3`, 10% infiltration, instant runoff, and a 1 km × 1 km floodpoint. Work out the rainfall threshold for one meter of water at the floodpoint.

In [ ]:
cell_area_m2 = 1_000_000  # 1 km × 1 km
flood_depth_m = 1.0
runoff_fraction = 0.9
if area_km2 == 0:
    raise ValueError("The watershed mask is empty; inspect the flow algorithm")
flood_volume_m3 = cell_area_m2 * flood_depth_m
uniform_rain_threshold_m = flood_volume_m3 / (area_km2 * cell_area_m2 * runoff_fraction)
print(f"Uniform rainfall threshold: {uniform_rain_threshold_m:.4f} m "
      f"({uniform_rain_threshold_m * 1000:.1f} mm)")


### Questions: uniform monsoon

> How much rain can the region receive before the 1 km x 1 km floodpoint is inundated with one meter of water?

**Response:**

> You might wish to comment on this assumption. How strong is it and why?

**Response:**

### Extra credit, part 3

> Describe how you would write a program so that the lowest region of the watershed ”filled up” with water realistically, with water spilling over into other grids.

**Response:**

### Spatially varying ten-day rainstorm

Generate `monsoon` with the shape of `topo3` and rainfall uniformly distributed from 0 to 1 meter. Apply the `shed` mask and 0.9 runoff fraction; sum watershed runoff, calculate the average ten-day flow rate through a 1 km wide channel under the stated steady-state assumption, and overlay `totalrain` on topography.

The notebook computes volumetric discharge in m³/s. Channel width alone does not specify water depth or speed.

In [ ]:
rng = np.random.default_rng(0)  # Reproducible example storm.
monsoon = rng.uniform(0, 1, size=topo3.shape)  # Meters over ten days.
totalrain = shed * monsoon  # MATLAB shed .* monsoon, before infiltration.
runoff_depth = 0.9 * totalrain
runoff_volume_m3 = runoff_depth.sum() * cell_area_m2
seconds_in_ten_days = 10 * 24 * 60 * 60
discharge_m3_s = runoff_volume_m3 / seconds_in_ten_days
print(f"Rain on watershed: {totalrain.sum() * cell_area_m2:,.0f} m³")
print(f"Runoff volume: {runoff_volume_m3:,.0f} m³")
print(f"Average discharge over ten days: {discharge_m3_s:,.2f} m³/s")
print("A 1 km channel width alone does not determine water depth or velocity.")
fig, ax = plt.subplots(figsize=(9, 5))
ax.imshow(topo3, origin="lower", cmap="terrain", alpha=0.6, interpolation="nearest")
image = ax.imshow(np.ma.masked_where(shed == 0, totalrain), origin="lower",
                  cmap="Blues", alpha=0.8, interpolation="nearest")
ax.set(title="Ten-day rainfall within the watershed", xlabel="Column", ylabel="Row")
plt.colorbar(image, ax=ax, label="Rainfall (m)")
plt.show()


### Results: ten-day monsoon

> Figure out how much of that rainfall has landed in our watershed.

**Response:**

> Find what the average flow rate (over the ten days) of water through our 1 km wide channel at the floodpoint must have been, assuming a steady state is reached.

**Response:**